# SI Figure 5

Compare slopes based on initial and final richness.

Input: generated/simulations.csv and generated/run_config.json. Run 00_generate_simulations.ipynb first. Set DATA_DIR for a different run.

Fit separate slopes for total, producer, and consumer biomass.

## Settings

Change DATA_DIR and SAVE in the next cell.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

DATA_DIR = Path('generated')
SAVE = True
FIGURE_DIR = Path('figures')
FOCUS_DEGREE = 1.0
FOCUS_STRENGTH = 0.5
simulations = pd.read_csv(DATA_DIR / 'simulations.csv')
config = json.loads((DATA_DIR / 'run_config.json').read_text(encoding='utf-8'))
FUNCTION_COL = 'biomass_for_bef'
if config['profile'] != 'final_10000':
    raise ValueError('Use data from the final_10000 simulation notebook')

def slope_table(df, x='N', y=None, groups=('degree','strength')):
    y = FUNCTION_COL if y is None else y
    rows = []
    for keys, part in df.groupby(list(groups)):
        if part[x].nunique() < 2:
            continue
        fit = linregress(part[x], part[y])
        rows.append(dict(zip(groups, keys if isinstance(keys, tuple) else (keys,))) |
                    dict(slope=fit.slope, intercept=fit.intercept, stderr=fit.stderr,
                         n=len(part), richness_levels=part[x].nunique()))
    return pd.DataFrame(rows)

def set_strength_axis(ax, values):
    lo, hi = float(np.min(values)), float(np.max(values))
    ticks = sorted(set([lo, hi] + [float(x) for x in np.linspace(.2, 1, 5) if lo < x < hi]))
    ax.set_xlim(lo, hi)
    ax.set_xticks(ticks, [f'{x:g}' for x in ticks])

def finish(fig, stem):
    fig.tight_layout()
    if SAVE:
        FIGURE_DIR.mkdir(exist_ok=True)
        fig.savefig(FIGURE_DIR / f'{stem}.png', dpi=300, bbox_inches='tight')
        fig.savefig(FIGURE_DIR / f'{stem}.pdf', bbox_inches='tight')
    plt.show()


## Figure

In [ ]:
data = simulations.copy()
data['consumer_biomass'] = data[FUNCTION_COL] - data.md0
data['consumer_initial'] = data.N - data.d0
data['consumer_final'] = data.richness_final - data.d0_final
panels = [('Total biomass', 'N', 'richness_final', FUNCTION_COL, 'magma_r', 1),
          ('Producer biomass', 'd0', 'd0_final', 'md0', 'Blues_r', .5),
          ('Consumer biomass', 'consumer_initial', 'consumer_final', 'consumer_biomass', 'Oranges_r', .5)]
degrees = np.sort(data.degree.unique())
strengths = np.sort(data.strength.unique())
fig, axes = plt.subplots(3, 2, figsize=(9, 11), sharex=True, sharey=True)
for row, (title, x_initial, x_final, y, cmap, vmax) in enumerate(panels):
    for col, (x, label) in enumerate([(x_initial, 'Initial'), (x_final, 'Final')]):
        table = slope_table(data, x=x, y=y)
        grid = (table.pivot(index='degree', columns='strength', values='slope')
                if len(table) else pd.DataFrame()).reindex(index=degrees, columns=strengths)
        ax = axes[row, col]
        image = ax.pcolormesh(strengths, degrees, grid.to_numpy(), shading='nearest',
                              cmap=cmap, vmin=0, vmax=vmax)
        ax.set_title(f'{"abcdef"[2*row+col]}  {label} richness: {title}')
        ax.set(xlabel='Interaction strength, max(omega)', ylabel='Connectance, mean degree')
        set_strength_axis(ax, strengths)
        fig.colorbar(image, ax=ax, fraction=.045, label='BEF slope')
        print(title, label.lower(), 'richness: estimable cells', len(table))
fig.tight_layout(); finish(fig, 'si_figure_5')